# Template Example Notebook

This is a template notebook. The first heading should be the title of what notebook is about. For example, if it is a project on neo4j tutorial the heading should be `Project Title`.

- Add description of what the notebook does.
- Point to references, e.g. (neo4j.example.md)
- Add citations.
- Keep the notebook flow clear.
- Comments should be imperative and have a period at the end.
- Your code should be well commented.

The name of this notebook should in the following format:
- if the notebook is exploring `pycaret API`, then it is `pycaret.example.ipynb`

Follow the reference to write notebooks in a clear manner: https://github.com/causify-ai/helpers/blob/master/docs/coding/all.jupyter_notebook.how_to_guide.md

In [21]:
# %load_ext autoreload
# %autoreload 2
# %matplotlib inline

In [22]:
# import logging
# # Import libraries in this section.
# # Avoid imports like import *, from ... import ..., from ... import *, etc.

# import helpers.hdbg as hdbg
# import helpers.hnotebook as hnotebo

In [23]:
# hdbg.init_logger(verbosity=logging.INFO)

# _LOG = logging.getLogger(__name__)

# hnotebo.config_notebook()

## Make the notebook flow clear
Each notebook needs to follow a clear and logical flow, e.g:
- Load data
- Compute stats
- Clean data
- Compute stats
- Do analysis
- Show results




#############################################################################
Template
#############################################################################

In [24]:
# class Template:
#     """
#     Brief imperative description of what the class does in one line, if needed.
#     """

#     def __init__(self):
#         pass

#     def method1(self, arg1: int) -> None:
#         """
#         Brief imperative description of what the method does in one line.

#         You can elaborate more in the method docstring in this section, for e.g. explaining
#         the formula/algorithm. Every method/function should have a docstring, typehints and include the
#         parameters and return as follows:

#         :param arg1: description of arg1
#         :return: description of return
#         """
#         # Code bloks go here.
#         # Make sure to include comments to explain what the code is doing.
#         # No empty lines between code blocks.
#         pass


# def template_function(arg1: int) -> None:
#     """
#     Brief imperative description of what the function does in one line.

#     You can elaborate more in the function docstring in this section, for e.g. explaining
#     the formula/algorithm. Every function should have a docstring, typehints and include the
#     parameters and return as follows:

#     :param arg1: description of arg1
#     :return: description of return
#     """
#     # Code bloks go here.
#     # Make sure to include comments to explain what the code is doing.
#     # No empty lines between code blocks.
#     pass

## The flow should be highlighted using headings in markdown
```
# Level 1
## Level 2
### Level 3
```

# Description

This notebook implements a full time series forecasting pipeline using the
Darts library to predict S&P 500 index prices and identify which market
sectors will outperform in the near future.

The notebook covers data collection for S&P 500, 11 sector ETFs, and 22
macroeconomic dimensions including yield curve, VIX, CPI, Fed rate, oil,
gold, and dollar index. It performs data preprocessing with release-date-aware
forward fill, feature engineering including technical indicators, calendar
features, and event flags for FOMC meetings, CPI release dates, and US
holidays. It trains the full Darts model suite across baseline, statistical,
probabilistic, and machine learning model groups, applies SHAP based feature
selection, performs hyperparameter tuning, builds ensemble models, and
benchmarks results against Facebook Prophet and Statsmodels. The sector
rotation engine recommends which sectors to rotate into based entirely on
model predictions, historical correlations, and risk adjusted scores.

References:
- Darts documentation: https://unit8co.github.io/darts/
- FRED API documentation: https://fred.stlouisfed.org/docs/api/fred/
- yfinance documentation: https://pypi.org/project/yfinance/
- Prophet documentation: https://facebook.github.io/prophet/

# Imports

In [25]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import logging
import os
import warnings

import darts
import darts.dataprocessing.transformers
import darts.metrics
import darts.models
import darts.timeseries
import dotenv
import fredapi
import holidays
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import prophet
import seaborn as sns
import shap
import sklearn.preprocessing
import statsmodels.tsa.arima.model
import statsmodels.tsa.holtwinters
import tqdm
import utils
import yfinance as yf

warnings.filterwarnings("ignore")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [26]:
# Configure the logger to track notebook execution.
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
)
_LOG = logging.getLogger(__name__)
# Log the environment setup confirmation and Darts version.
_LOG.info("Environment configured successfully.")
_LOG.info("Darts version: %s", darts.__version__)

2026-04-03 18:03:24,760 - INFO - Environment configured successfully.
2026-04-03 18:03:24,760 - INFO - Darts version: 0.43.0


# Configuration

In [27]:
# Define the date range for all data downloads.
START_DATE = "2018-01-01"
END_DATE = "2024-12-31"
# Define the S&P 500 ticker symbol.
SP500_TICKER = "^GSPC"
# Define all 11 sector ETF ticker symbols.
SECTOR_TICKERS = [
    "XLK",
    "XLV",
    "XLF",
    "XLE",
    "XLY",
    "XLP",
    "XLI",
    "XLU",
    "XLB",
    "XLRE",
    "XLC",
]
# Define sector names mapped to their ticker symbols.
SECTOR_NAMES = {
    "XLK": "Technology",
    "XLV": "Healthcare",
    "XLF": "Financials",
    "XLE": "Energy",
    "XLY": "Consumer Discretionary",
    "XLP": "Consumer Staples",
    "XLI": "Industrials",
    "XLU": "Utilities",
    "XLB": "Materials",
    "XLRE": "Real Estate",
    "XLC": "Communication Services",
}
# Define daily macro indicator ticker symbols from yfinance.
DAILY_MACRO_TICKERS = {
    "VIX": "^VIX",
    "TNX": "^TNX",
    "IRX": "^IRX",
    "OIL": "CL=F",
    "GOLD": "GC=F",
    "DXY": "DX-Y.NYB",
}
# Define monthly macro indicator codes from FRED API.
MONTHLY_MACRO_CODES = {
    "CPI": "CPIAUCSL",
    "CORE_CPI": "CPILFESL",
    "FED_RATE": "FEDFUNDS",
    "UNEMPLOYMENT": "UNRATE",
    "NFP": "PAYEMS",
    "RETAIL_SALES": "RSAFS",
    "INDUSTRIAL_PROD": "INDPRO",
    "PCE": "PCEPI",
    "PPI": "PPIACO",
}
# Define the FRED daily indicator code for breakeven inflation.
BREAKEVEN_INFLATION_CODE = "T10YIE"
# Define the data directory path for saving raw CSV files.
DATA_DIR = "data"
# Define the test set size in trading days.
TEST_SIZE = 60
# Define the validation set size in trading days.
VAL_SIZE = 60
# Define the forecast horizon in trading days.
FORECAST_HORIZON = 30
# Create the data directory if it does not exist.
os.makedirs(DATA_DIR, exist_ok=True)
_LOG.info("Configuration loaded successfully.")
_LOG.info("Date range: %s to %s", START_DATE, END_DATE)
_LOG.info("Sectors: %s", list(SECTOR_NAMES.values()))

2026-04-03 18:03:24,777 - INFO - Configuration loaded successfully.
2026-04-03 18:03:24,777 - INFO - Date range: 2018-01-01 to 2024-12-31
2026-04-03 18:03:24,777 - INFO - Sectors: ['Technology', 'Healthcare', 'Financials', 'Energy', 'Consumer Discretionary', 'Consumer Staples', 'Industrials', 'Utilities', 'Materials', 'Real Estate', 'Communication Services']


# Data collection

This section downloads all required data from Yahoo Finance and the FRED
API and saves it as raw CSV files in the `data` directory. The data
includes S&P 500 historical prices, 11 sector ETF prices, 6 daily
macroeconomic indicators, and 10 monthly macroeconomic indicators covering
the period from 2018 to 2024.

In [28]:
# Load environment variables from the .env file.
dotenv.load_dotenv()
# Read the FRED API key from the environment variables.
FRED_API_KEY = os.environ.get("FRED_API_KEY")
# Verify the FRED API key was loaded successfully.
if FRED_API_KEY is None:
    raise ValueError("FRED_API_KEY not found in .env file.")
_LOG.info("FRED API key loaded successfully.")

2026-04-03 18:03:24,794 - INFO - FRED API key loaded successfully.


In [29]:
# Download S&P 500 historical price data.
sp500 = utils.download_sp500(SP500_TICKER, START_DATE, END_DATE)
sp500.head(5)

2026-04-03 18:03:24,809 - INFO - Downloading S&P 500 data from 2018-01-01 to 2024-12-31.
2026-04-03 18:03:24,827 - INFO - Downloaded 1760 rows of S&P 500 data.


Price,Close,High,Low,Open,Volume
Date,,,,,
2018-01-02,2695.810059,2695.889893,2682.360107,2683.729980,3397430000
2018-01-03,2713.060059,2714.370117,2697.770020,2697.850098,3544030000
2018-01-04,2723.989990,2729.290039,2719.070068,2719.310059,3697340000
2018-01-05,2743.149902,2743.449951,2727.919922,2731.330078,3239280000
2018-01-08,2747.709961,2748.510010,2737.600098,2742.669922,3246160000


In [30]:
# Save the raw S&P 500 data to CSV for reproducibility.
utils.save_data(sp500, "sp500_raw.csv", DATA_DIR)

2026-04-03 18:03:24,847 - INFO - Saved 1760 rows to data/sp500_raw.csv.


In [31]:
# Download historical price data for all 11 sector ETFs.
sectors = utils.download_sectors(SECTOR_TICKERS, START_DATE, END_DATE)
sectors.head(5)

2026-04-03 18:03:24,861 - INFO - Downloading 11 sector ETFs.
2026-04-03 18:03:25,009 - INFO - Downloaded 1760 rows for 11 sectors.


,XLK,XLV,XLF,XLE,XLY,XLP,XLI,XLU,XLB,XLRE,XLC
Date,,,,,,,,,,,
2018-01-02,29.872929,72.723343,23.884140,25.747257,46.275227,45.314564,66.254837,20.132587,25.995674,24.837584,NaN
2018-01-03,30.122103,73.419174,24.012465,26.132860,46.487701,45.298534,66.611694,19.974422,26.177759,24.845160,NaN
2018-01-04,30.274364,73.523544,24.234877,26.290604,46.640125,45.426777,67.099129,19.808548,26.406425,24.420465,NaN
2018-01-05,30.592756,74.149780,24.303310,26.280087,47.009624,45.627136,67.560440,19.800835,26.618153,24.473553,NaN
2018-01-08,30.708111,73.880157,24.269094,26.437824,47.065044,45.739338,67.838966,19.985994,26.656265,24.640400,NaN


In [32]:
# Save the raw sector ETF data to CSV for reproducibility.
utils.save_data(sectors, "sectors_raw.csv", DATA_DIR)

2026-04-03 18:03:25,034 - INFO - Saved 1760 rows to data/sectors_raw.csv.


In [33]:
# Download daily macroeconomic indicators from Yahoo Finance.
macro_daily = utils.download_daily_macro(
    DAILY_MACRO_TICKERS, START_DATE, END_DATE
)
macro_daily.head(5)

2026-04-03 18:03:25,047 - INFO - Downloading 6 daily macro indicators.
2026-04-03 18:03:25,131 - INFO - Downloaded 1760 rows for 6 daily macro indicators.


,VIX,TNX,IRX,OIL,GOLD,DXY
Date,,,,,,
2018-01-02,9.77,2.465,1.378,60.369999,1313.699951,91.849998
2018-01-03,9.15,2.447,1.370,61.630001,1316.199951,92.160004
2018-01-04,9.22,2.453,1.370,62.009998,1319.400024,91.849998
2018-01-05,9.22,2.476,1.370,61.439999,1320.300049,91.949997
2018-01-08,9.52,2.480,1.380,61.730000,1318.599976,92.330002


In [34]:
# Save the raw daily macro indicator data to CSV for reproducibility.
utils.save_data(macro_daily, "macro_daily_raw.csv", DATA_DIR)

2026-04-03 18:03:25,151 - INFO - Saved 1760 rows to data/macro_daily_raw.csv.


In [35]:
# Download monthly macroeconomic indicators from the FRED API.
macro_monthly = utils.download_monthly_macro(
    MONTHLY_MACRO_CODES,
    BREAKEVEN_INFLATION_CODE,
    START_DATE,
    END_DATE,
    FRED_API_KEY,
)
macro_monthly.head(3)

2026-04-03 18:03:25,164 - INFO - Downloading 9 monthly macro indicators from FRED.
2026-04-03 18:03:27,333 - INFO - Downloading 10Y breakeven inflation from FRED.
2026-04-03 18:03:27,757 - INFO - Downloaded 1827 rows for 10 monthly macro indicators.


,CPI,CORE_CPI,FED_RATE,UNEMPLOYMENT,NFP,RETAIL_SALES,INDUSTRIAL_PROD,PCE,PPI,BREAKEVEN
Date,,,,,,,,,,
2018-01-01,248.859,255.204,1.41,4.0,147660.0,481414.0,101.4625,101.199,197.9,NaN
2018-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.00
2018-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.98


In [36]:
# Save the raw monthly macro indicator data to CSV for reproducibility.
utils.save_data(macro_monthly, "macro_monthly_raw.csv", DATA_DIR)

2026-04-03 18:03:27,776 - INFO - Saved 1827 rows to data/macro_monthly_raw.csv.
